# STR_EPF (Star tracker energetic particle flux)

> Abstract: The `SW_OPER_STRxEPF_1B` products give the flux of energetic particles (protons above about 100 MeV) detected as spurious bright pixels in the images from the three star tracker Camera Head Units (CHUs) on each Swarm spacecraft. This notebook shows how to access the products with viresclient, how the data from the three CHUs is laid out, how to use the quality flags, and how the products capture the solar energetic particle event of 11 May 2024.

In [ ]:
%load_ext watermark
%watermark -i -v -p viresclient,pandas,xarray,matplotlib

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from viresclient import SwarmRequest

These products are currently only available from the VirES staging server, so we point viresclient to it explicitly. If you have not used the staging server before, you will be prompted to set an access token for it (see the [viresclient documentation](https://viresclient.readthedocs.io/en/latest/config_details.html)).

In [ ]:
SERVER_URL = "https://staging.vires.services/ows"

## Product information

Each Swarm spacecraft carries three star tracker CHUs. Besides determining the attitude, the CHU images record hits from energetic particles that penetrate the camera shielding. The STR_EPF products count these hits (`EPD`), correct them for star density and CHU temperature, and convert them to a flux estimate (`EPF`, in cm<sup>-2</sup> s<sup>-1</sup>), at a 2 Hz cadence for each CHU.

Documentation:
- https://swarmhandbook.earth.esa.int/catalogue/SW_STRXEPF_1B

The products are provided in two forms:

- `SW_OPER_STRxEPF_1B` (x = A, B, C): one record per time, with each per-CHU variable holding three values (one for each CHU). viresclient gives these a `CHU` dimension (labelled 1, 2, 3) in xarray, or splits them into columns `<var>_1`, `<var>_2`, `<var>_3` in pandas with `expand=True`
- `SW_OPER_STRxEPF_1B:flattened`: one record per time *and* CHU, with an extra `SensorId` variable (1, 2, 3) identifying the CHU

Near-real-time versions of both are available as `SW_FAST_STRxEPF_1B` (see [FAST data](06a1_FAST-Intro.ipynb)).

In [ ]:
request = SwarmRequest(SERVER_URL)
request.available_collections("STR_EPF", details=False)

In [ ]:
for collection in ("SW_OPER_STRAEPF_1B", "SW_OPER_STRAEPF_1B:flattened"):
    print(f"{collection}:\n{request.available_measurements(collection)}\n")

| Variable | Units | Description |
|---|---|---|
| `EPF` | cm<sup>-2</sup> s<sup>-1</sup> | Energetic particle flux (NaN if `Flags_EPF` >= 16) |
| `EPD` | - | Energetic particle detections (raw count) |
| `EPD_Dens_corr` | - | Star density correction of particle count |
| `EPD_T_corr` | - | Temperature correction of particle count |
| `T_CHU` | °C | CHU temperature |
| `RA`, `Dec` | deg | Right ascension and declination of the CHU boresight |
| `Flags_EPF` | - | Quality flags (see below) |
| `SensorId` | - | CHU identifier (`:flattened` collections only) |

`Flags_EPF` is a bit field. Each bit marks a condition affecting the flux estimate:

| Bit value | Meaning |
|---|---|
| 0 | Energetic particle flux estimate nominal |
| 1 | Star density above limit in field of view |
| 2 | CHU temperature above limit |
| 4 | Gain control not stabilized |
| 8 | Raw particle count above limit |
| 16 | Big bright object (BBO) detected |
| 32 | CHU in high-rate mode (image smear) |
| 64 | Non-valid attitude sample |
| 128 | On-board processing not completed |

Check the time coverage of the products:

In [ ]:
request.available_times("SW_OPER_STRAEPF_1B")

## Fetching data for a single day

We fetch one day of full-resolution data from Swarm Alpha, together with the magnetic quasi-dipole latitude (`QDLat`) and magnetic local time (`MLT`). We choose 11 May 2024, during the strong geomagnetic storm of May 2024, when a solar energetic particle (SEP) event delivered protons with energies high enough to be detected at ground level (ground level enhancement GLE 74).

In [ ]:
request = SwarmRequest(SERVER_URL)
request.set_collection("SW_OPER_STRAEPF_1B")
request.set_products(
    measurements=["EPF", "EPD", "T_CHU", "Flags_EPF"],
    auxiliaries=["QDLat", "MLT"],
)
data = request.get_between(dt.datetime(2024, 5, 11), dt.datetime(2024, 5, 12))
ds = data.as_xarray()
ds

The per-CHU variables share a `CHU` dimension, labelled 1, 2, 3, so we can select or reduce over the CHUs by name, e.g. `ds["EPF"].sel(CHU=2)` or `ds["EPF"].mean(dim="CHU")`.

To work with pandas instead, use `expand=True` to split each per-CHU variable into one column per CHU:

In [ ]:
data.as_dataframe(expand=True).head()

Plot the flux from each CHU over a few hours, alongside the QD latitude. Each sample is based on only a few particle detections in a single image, so the 2 Hz data are dominated by counting noise. Averaging over the three CHUs and a 2-minute window (240 samples) brings out the variation along the orbit.

In [ ]:
subset = ds.sel(Timestamp=slice("2024-05-11T00:00", "2024-05-11T06:00"))
EPF_smoothed = subset["EPF"].mean(dim="CHU").rolling(Timestamp=240, center=True).mean()

fig, axes = plt.subplots(nrows=2, sharex=True, figsize=(12, 6))
subset["EPF"].plot.line(x="Timestamp", hue="CHU", ax=axes[0], linewidth=0.5, alpha=0.5)
EPF_smoothed.plot(ax=axes[0], color="black", label="2-minute mean, all CHUs")
subset["QDLat"].plot(ax=axes[1], color="black")
axes[0].set_xlabel("")
axes[0].set_ylim(0, 150)
axes[0].set_title("Swarm Alpha")
axes[1].set_title("")
axes[1].set_yticks([-90, -45, 0, 45, 90])
axes[1].grid();

### Quality flags

Since `Flags_EPF` is a bit field, we test for each bit with a bitwise AND. Count the samples during the day with each bit set, in one column per CHU:

In [ ]:
FLAG_BITS = {
    1: "Star density above limit",
    2: "CHU temperature above limit",
    4: "Gain control not stabilized",
    8: "Raw particle count above limit",
    16: "Big bright object detected",
    32: "CHU in high-rate mode",
    64: "Non-valid attitude sample",
    128: "On-board processing not completed",
}

flags = ds["Flags_EPF"]
flag_counts = {"0: Nominal": (flags == 0).sum(dim="Timestamp").to_pandas()}
for bit, meaning in FLAG_BITS.items():
    flag_counts[f"{bit}: {meaning}"] = (
        ((flags & bit) > 0).sum(dim="Timestamp").to_pandas()
    )
pd.DataFrame(flag_counts).T

Where any flag of 16 or above is set, no flux could be estimated and `EPF` is already NaN. The lower bits mark conditions that may degrade the estimate.

Here, the "CHU temperature above limit" flag is set almost all day for CHUs 1 and 3, so keeping only fully nominal samples (`Flags_EPF == 0`) would discard nearly all the data from those two CHUs:

In [ ]:
EPF_nominal = ds["EPF"].where(ds["Flags_EPF"] == 0)
pd.DataFrame(
    {
        "Valid EPF samples": ds["EPF"].notnull().sum(dim="Timestamp").to_pandas(),
        "Nominal EPF samples": EPF_nominal.notnull().sum(dim="Timestamp").to_pandas(),
    }
)

How strictly to filter depends on your application. In the rest of this notebook, we use all samples where `EPF` is valid (not NaN).

## Using the flattened collections

The `:flattened` collections give one row per CHU and time, which is convenient for tabular work with pandas. Here we fetch a week of data spanning the event, downsampled to one sample every 10 seconds to make the request lighter.

In [ ]:
def fetch_flattened(start_time, end_time, spacecraft="A"):
    request = SwarmRequest(SERVER_URL)
    request.set_collection(f"SW_OPER_STR{spacecraft}EPF_1B:flattened")
    request.set_products(
        measurements=["EPF", "Flags_EPF", "SensorId"],
        auxiliaries=["QDLat", "MLT"],
        sampling_step="PT10S",
    )
    data = request.get_between(start_time, end_time)
    return data.as_dataframe()


start_time = dt.datetime(2024, 5, 8)
end_time = dt.datetime(2024, 5, 15)
df = fetch_flattened(start_time, end_time, spacecraft="A")
df.head()

Each timestamp appears three times, once for each `SensorId`, so we can use pandas groupby to compare the CHUs:

In [ ]:
df.groupby("SensorId")["EPF"].describe()

Average the flux from all CHUs over the week in bins of latitude and longitude. The main feature is the South Atlantic Anomaly (SAA), where the inner radiation belt reaches down to Swarm altitudes. Bands of enhanced flux are also visible at high latitudes, especially in the southern hemisphere.

In [ ]:
lat_bins = np.arange(-90, 91, 5)
lon_bins = np.arange(-180, 181, 10)
EPF_map = (
    df.groupby(
        [pd.cut(df["Latitude"], lat_bins), pd.cut(df["Longitude"], lon_bins)],
        observed=False,
    )["EPF"]
    .mean()
    .unstack()
)

fig, ax = plt.subplots(figsize=(10, 5))
mesh = ax.pcolormesh(lon_bins, lat_bins, EPF_map.values, cmap="inferno", vmax=40)
fig.colorbar(mesh, ax=ax, label="Mean EPF [cm$^{-2}$ s$^{-1}$]", extend="max")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
ax.set_title(f"Swarm Alpha, all CHUs, {start_time:%Y-%m-%d} to {end_time:%Y-%m-%d}");

Compare the mean flux as a function of QD latitude on a quiet day before the storm with the day of the SEP event. On 11 May, the flux is enhanced above about 55° QD latitude in both hemispheres: the SEP protons reach Swarm altitudes only at high magnetic latitudes, where the geomagnetic field gives less shielding.

In [ ]:
qdlat_bins = np.arange(-90, 91, 2)
qdlat_centres = qdlat_bins[:-1] + 1

fig, ax = plt.subplots(figsize=(10, 4))
for day, label in (
    ("2024-05-08", "8 May (quiet)"),
    ("2024-05-11", "11 May (SEP event)"),
):
    df_day = df.loc[day]
    profile = df_day.groupby(pd.cut(df_day["QDLat"], qdlat_bins), observed=False)[
        "EPF"
    ].mean()
    ax.plot(qdlat_centres, profile.values, label=label)
ax.set_xlabel("QD Latitude [deg]")
ax.set_ylabel("Mean EPF [cm$^{-2}$ s$^{-1}$]")
ax.set_xlim(-90, 90)
ax.set_title("Swarm Alpha, all CHUs")
ax.legend()
ax.grid();

## The time evolution of the SEP event

To follow the event, we take the hourly mean flux over the polar caps (|QDLat| > 70°) from all three spacecraft. All three show the enhancement on 11 May. The background level differs between spacecraft, and the event stands out most clearly against the low background on Swarm Bravo.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for spacecraft in "ABC":
    df_sc = (
        df if spacecraft == "A" else fetch_flattened(start_time, end_time, spacecraft)
    )
    polar = df_sc[df_sc["QDLat"].abs() > 70]
    polar["EPF"].resample("1h").mean().plot(ax=ax, label=f"Swarm {spacecraft}")
ax.set_ylabel("Mean polar cap EPF\n[cm$^{-2}$ s$^{-1}$]")
ax.set_xlabel("")
ax.legend()
ax.grid();

## Near-real-time (FAST) data

The `SW_FAST_STRxEPF_1B` collections provide the same data shortly after acquisition, which is useful for monitoring current conditions. They have the same variables as the OPER products, so the code above works if you change the collection name. FAST data are only kept for recent months:

In [ ]:
request = SwarmRequest(SERVER_URL)
fast_times = request.available_times("SW_FAST_STRAEPF_1B")
print(
    f"FAST data available from {fast_times['starttime'].min()} to {fast_times['endtime'].max()}"
)

Fetch the latest 6 hours of FAST data from Swarm Alpha and plot the 2-minute mean flux from each CHU:

In [ ]:
end_time = fast_times["endtime"].max().to_pydatetime()
start_time = end_time - dt.timedelta(hours=6)

request = SwarmRequest(SERVER_URL)
request.set_collection("SW_FAST_STRAEPF_1B:flattened")
request.set_products(
    measurements=["EPF", "Flags_EPF", "SensorId"],
    auxiliaries=["QDLat"],
)
df_fast = request.get_between(start_time, end_time).as_dataframe()

fig, axes = plt.subplots(nrows=2, sharex=True, figsize=(12, 5))
for sensor_id, df_chu in df_fast.groupby("SensorId"):
    df_chu["EPF"].rolling("2min", center=True).mean().plot(
        ax=axes[0], label=f"CHU {sensor_id}"
    )
df_fast["QDLat"].plot(ax=axes[1], color="black")
axes[0].set_ylabel("EPF [cm$^{-2}$ s$^{-1}$]")
axes[0].legend()
axes[1].set_ylabel("QD Latitude [deg]")
axes[1].set_xlabel("")
axes[0].set_title("Swarm Alpha (FAST): latest 6 hours, 2-minute mean");